In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install torch torchvision flask flask-ngrok opencv-python numpy Pillow


In [3]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define dataset path
data_dir = "/content/drive/MyDrive/project/dataset_blood_group"

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet-50 input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ResNet normalization
])

# Load dataset
dataset = ImageFolder(root=data_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Display class names
print("Classes:", dataset.classes)


Classes: ['train', 'val']


In [4]:
import torchvision.models as models
import torch.nn as nn

# Load pre-trained ResNet-50 model
model = models.resnet50(pretrained=True)

# Modify the final layer for 8 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 8)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 125MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Define project directory
PROJECT_DIR = "/content/drive/MyDrive/project/"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.models as models

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [13]:
# Define loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.fc.parameters(),
    lr=0.0001,
    weight_decay=1e-4
)

# Learning rate scheduler
# Reduces learning rate when validation loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

In [14]:
import torchvision.models as models
from torchvision.models import ResNet50_Weights

# Load ResNet50 with pre-trained weights
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

# Modify the final layer for 8-class classification
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 8)  # 8 classes (A+, A-, B+, B-, O+, O-, AB+, AB-)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [15]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)  # Better for generalization
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Reduce LR every 10 epochs


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader, random_split

# ✅ Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ✅ Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

# ✅ Define transformations for image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize for ResNet50
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# ✅ Load Dataset
dataset_path = "/content/drive/MyDrive/project/dataset_blood_group"  # Change path as needed
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

# ✅ Split dataset into training (80%) and validation (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# ✅ Create DataLoaders
batch_size = 16  # Reduce batch size if training is slow
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ✅ Check if DataLoader is working
data_iter = iter(train_loader)
sample_images, sample_labels = next(data_iter)
print(f"Sample batch shape: {sample_images.shape}")  # Should be (batch_size, 3, 224, 224)
print(f"Sample labels: {sample_labels}")  # Should be tensor([values])

# ✅ Load Pretrained ResNet50 Model
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 8)  # 8 blood group classes
model = model.to(device)

# ✅ Define Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)  # Reduce LR every 5 epochs

# ✅ Training Loop
num_epochs = 10  # Start with 10 epochs to check progress
print("Starting Training...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs} ----------------------------")

    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 10 == 0:  # Print progress every 10 batches
            print(f"Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")

    # ✅ Validation Loop
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total * 100

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.2f}%")

    scheduler.step()  # Adjust learning rate

    # ✅ Save model after every epoch
    model_save_path = f"/content/drive/MyDrive/project/blood_group_model_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), model_save_path)
    print(f"Model saved at {model_save_path}")

print("Training Completed!")


Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sample batch shape: torch.Size([16, 3, 224, 224])
Sample labels: tensor([0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0])
Starting Training...

Epoch 1/10 ----------------------------
Batch 0/400 - Loss: 2.1869


In [ ]:
import os

image_path = "/content/drive/MyDrive/Colab Notebooks/mini project/sample_fingerprint.BMP"

if os.path.exists(image_path):
    print("✅ File found!")
else:
    print("❌ File NOT found! Check the path or upload the file.")


In [ ]:
from PIL import Image

# Load Test Image
image_path = "/content/drive/MyDrive/Colab Notebooks/mini project/sample_fingerprint.BMP"
image = Image.open(image_path).convert("RGB")

# Show the image to confirm it's loaded correctly
image.show()


In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# Define Device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Pretrained ResNet-50 Model
model = models.resnet50(pretrained=False)  # Ensure same architecture
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 8)  # 8 classes for blood groups
model = model.to(device)  # Move to device

# Load Trained Weights
model_path = "/content/drive/MyDrive/Colab Notebooks/mini project/blood_group_model_epoch_40.pth"
model.load_state_dict(torch.load(model_path, map_location=device))  # Load on the correct device
model.eval()

# Load Test Image
image_path = "/content/drive/MyDrive/Colab Notebooks/mini project/sample_fingerprint.BMP"
image = Image.open(image_path).convert("RGB")

# Define Transformations (Same as Used During Training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Adjust channels
])

# Apply Transform and Add Batch Dimension
image = transform(image).unsqueeze(0).to(device)

# Make Prediction
with torch.no_grad():
    output = model(image)
    _, predicted = torch.max(output, 1)

# Define Blood Group Labels
blood_groups = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]

# Print Prediction
print(f"Predicted Blood Group: {blood_groups[predicted.item()]}")


In [ ]:
pip install flask flask-cors torchvision


In [ ]:
!pip install pyngrok flask-ngrok

In [ ]:
!pip install flask flask-ngrok pyngrok

In [ ]:
!ngrok authtoken xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx


In [ ]:
import os

model_path = "/content/drive/MyDrive/Colab Notebooks/mini project/blood_group_model_epoch_40.pth"
print("Model Exists:", os.path.exists(model_path))


In [ ]:
model_path = "/content/drive/MyDrive/Colab Notebooks/mini project/blood_group_model_epoch_40.pth"
model.load_state_dict(torch.load(model_path, map_location=device))


In [ ]:
!pkill -f ngrok  # Kill any existing ngrok processes
from pyngrok import ngrok

# Re-open ngrok tunnel
public_url = ngrok.connect(5000)
print(" * ngrok tunnel:", public_url)


In [ ]:
!pkill -f flask  # Kill existing Flask process
!flask run --host=0.0.0.0 --port=5000 &


In [ ]:
%env FLASK_APP=app.py  # If your main file is named app.py


In [ ]:
!pkill -f flask
!pkill -f ngrok


In [ ]:
!pip install flask pyngrok


In [ ]:
!pkill -f ngrok  # Kill any existing ngrok process

from flask import Flask
from pyngrok import ngrok

app = Flask(__name__)

@app.route("/")
def home():
    return "Flask is Running Successfully!"

# ✅ Authenticate ngrok with your token (Only needed once)
NGROK_AUTH_TOKEN = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# ✅ Start Flask before opening ngrok tunnel
from threading import Thread
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

Thread(target=run_flask).start()

# ✅ Open ngrok tunnel
public_url = ngrok.connect(5000)
print("🔗 Public URL:", public_url)


In [ ]:
!pip install flask-cors


In [ ]:
import torch
import torchvision.models as models

class BloodGroupModel(torch.nn.Module):
    def __init__(self):
        super(BloodGroupModel, self).__init__()
        self.model = models.resnet50(weights=None)  # Make sure this matches the training architecture
        num_ftrs = self.model.fc.in_features
        self.model.fc = torch.nn.Linear(num_ftrs, 8)  # Adjust output layers correctly

    def forward(self, x):
        return self.model(x)

# Initialize model
model = BloodGroupModel()


In [8]:
state_dict = torch.load(MODEL_PATH, map_location=torch.device('cpu'))
model.load_state_dict(state_dict, strict=False)


NameError: name 'MODEL_PATH' is not defined

In [ ]:
state_dict = torch.load(MODEL_PATH, map_location=torch.device('cpu'))
model.load_state_dict(state_dict, strict=False)


In [ ]:
state_dict = torch.load(MODEL_PATH, map_location=torch.device('cpu'))

print("Saved Model Keys:\n", state_dict.keys())
print("\nCurrent Model Keys:\n", model.state_dict().keys())


In [ ]:
torch.save(model.state_dict(), 'blood_group_model.pth')


In [ ]:
model.load_state_dict(torch.load('blood_group_model.pth', map_location=torch.device('cpu')))


In [ ]:
import torch
import torchvision.models as models

model = models.resnet50(pretrained=False)
model.fc = torch.nn.Linear(2048, 8)  # Adjust for 8 classes
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
model.eval()


In [ ]:
import torch

# Load the model state_dict
state_dict = torch.load(MODEL_PATH, map_location=torch.device('cpu'))

# Remove "model." prefix if present
new_state_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}

# Load the corrected state dict
model.load_state_dict(new_state_dict)


In [ ]:
import torch
from PIL import Image
import torchvision.transforms as transforms
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the correct path to your image
img_path = "/content/drive/MyDrive/Colab Notebooks/mini project/sample_fingerprint.BMP"

# Load and preprocess the image
image = Image.open(img_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to match model input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Adjust if needed
])

image = transform(image).unsqueeze(0)  # Add batch dimension

# Ensure the model is in evaluation mode
model.eval()

# Perform inference
output = model(image)

# Get the predicted class index
predicted_class = torch.argmax(output, dim=1).item()

# Define blood group labels
blood_groups = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]

# Print the predicted blood group
print("Predicted Blood Group:", blood_groups[predicted_class])


In [ ]:
import torch
import torchvision.models as models

# Define the model architecture (ResNet-50)
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 8)  # 8 classes for blood groups

# Load the trained model
MODEL_PATH = "/content/drive/MyDrive/blood_group_model_epoch_30.pth"
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device("cpu")))
model.eval()

print("✅ Model loaded successfully!")


In [ ]:
sample_input = torch.randn(1, 3, 224, 224)  # Dummy input
output = model(sample_input)

blood_groups = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
predicted_class = torch.argmax(output, dim=1).item()

print("Predicted Blood Group:", blood_groups[predicted_class])


In [ ]:
pip install flask torch torchvision pillow werkzeug


In [ ]:
import os
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from flask import Flask, render_template, request, jsonify
from PIL import Image
from pyngrok import ngrok

# Set Paths (As Per Your Drive Structure)
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/mini project/blood_group_model_epoch_40.pth"
UPLOAD_FOLDER = "/content/drive/MyDrive/Colab Notebooks/mini project/uploads"
TEMPLATE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/mini project/templates"
STATIC_FOLDER = "/content/drive/MyDrive/Colab Notebooks/mini project/static"

# Ensure Uploads Directory Exists
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# Initialize Flask App
app = Flask(__name__, template_folder=TEMPLATE_FOLDER, static_folder=STATIC_FOLDER)
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER

# Ngrok Tunnel for Colab
public_url = ngrok.connect(5000).public_url
print(f"Public URL: {public_url}")

# Load Pretrained Model
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 8)
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device("cpu")))
model.eval()

# Blood Group Labels
blood_groups = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]

# Image Transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Home Page Route
@app.route("/")
def index():
    return render_template("index.html")

# Upload File Route
@app.route("/upload", methods=["POST"])
def upload():
    if "file" not in request.files:
        return jsonify({"error": "No file uploaded"}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No selected file"}), 400

    if file:
        filepath = os.path.join(UPLOAD_FOLDER, file.filename)
        file.save(filepath)
        return jsonify({"success": True, "filepath": filepath})

# Predict Blood Group
@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    filepath = data.get("filepath")

    if not filepath or not os.path.exists(filepath):
        return jsonify({"error": "Invalid file path"}), 400

    # Load Image
    image = Image.open(filepath).convert("RGB")
    image = transform(image).unsqueeze(0)

    # Model Prediction
    output = model(image)
    predicted_class = torch.argmax(output, dim=1).item()
    predicted_blood_group = blood_groups[predicted_class]

    return jsonify({"predicted_blood_group": predicted_blood_group})

if __name__ == "__main__":
    app.run(port=5000)


In [ ]:
import torch
import torchvision.models as models

# Define the Model Architecture (ResNet-50)
model = models.resnet50(pretrained=False)  # Initialize ResNet-50
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 8)  # Adjust output layer for 8 blood groups

# Load the saved state_dict
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))

# Set model to evaluation mode
model.eval()


In [ ]:
import torch
from flask import Flask, request, jsonify, render_template
from werkzeug.utils import secure_filename
from pyngrok import ngrok
import torchvision.transforms as transforms
from PIL import Image
import os

# Ngrok Tunnel for Colab
public_url = ngrok.connect(5000).public_url
print(f"Public URL: {public_url}")

# Flask App Configuration
TEMPLATE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/mini project/templates"
STATIC_FOLDER = "/content/drive/MyDrive/Colab Notebooks/mini project/static"
MODEL_PATH = "/content/drive/MyDrive/blood_group_model_epoch_30.pth"

app = Flask(__name__, template_folder=TEMPLATE_FOLDER, static_folder=STATIC_FOLDER)

# Load the trained model
model = torch.load(MODEL_PATH, map_location=torch.device('cpu'))
model.eval()

# Image Preprocessing Function
def preprocess_image(image_path):
    image = Image.open(image_path)

    # Convert grayscale to RGB (ResNet expects 3 channels)
    if image.mode != "RGB":
        image = image.convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize for ResNet
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize like training data
    ])

    image_tensor = transform(image).unsqueeze(0)  # Add batch dimension
    return image_tensor

# Route for Web App Interface
@app.route('/')
def home():
    return render_template('index.html')

# Route for Prediction
@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({"error": "No file uploaded"})

    file = request.files['file']
    filename = secure_filename(file.filename)
    file_path = os.path.join(STATIC_FOLDER, filename)
    file.save(file_path)

    # Preprocess the uploaded image
    image_tensor = preprocess_image(file_path)

    # Run prediction
    with torch.no_grad():
        output = model(image_tensor)
        predicted_class = torch.argmax(output).item()

    # Blood group mapping
    blood_groups = ['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-']
    predicted_blood_group = blood_groups[predicted_class]

    return jsonify({"Predicted Blood Group": predicted_blood_group})

if __name__ == '__main__':
    app.run(port=5000)
